# Load Data

In [1]:
from ISLP import load_data
boston = load_data('Boston')
boston.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     506 non-null    float64
 1   zn       506 non-null    float64
 2   indus    506 non-null    float64
 3   chas     506 non-null    int64  
 4   nox      506 non-null    float64
 5   rm       506 non-null    float64
 6   age      506 non-null    float64
 7   dis      506 non-null    float64
 8   rad      506 non-null    int64  
 9   tax      506 non-null    int64  
 10  ptratio  506 non-null    float64
 11  lstat    506 non-null    float64
 12  medv     506 non-null    float64
dtypes: float64(10), int64(3)
memory usage: 51.5 KB


In [2]:
boston.describe()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
count,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000
mean,3.613524,11.363636,11.136779,0.069170,0.554695,6.284634,68.574901,3.795043,9.549407,408.237154,18.455534,12.653063,22.532806
std,8.601545,23.322453,6.860353,0.253994,0.115878,0.702617,28.148861,2.105710,8.707259,168.537116,2.164946,7.141062,9.197104
min,0.006320,0.000000,0.460000,0.000000,0.385000,3.561000,2.900000,1.129600,1.000000,187.000000,12.600000,1.730000,5.000000
25%,0.082045,0.000000,5.190000,0.000000,0.449000,5.885500,45.025000,2.100175,4.000000,279.000000,17.400000,6.950000,17.025000
50%,0.256510,0.000000,9.690000,0.000000,0.538000,6.208500,77.500000,3.207450,5.000000,330.000000,19.050000,11.360000,21.200000
75%,3.677083,12.500000,18.100000,0.000000,0.624000,6.623500,94.075000,5.188425,24.000000,666.000000,20.200000,16.955000,25.000000
max,88.976200,100.000000,27.740000,1.000000,0.871000,8.780000,100.000000,12.126500,24.000000,711.000000,22.000000,37.970000,50.000000


In [3]:
X = boston.drop(columns=["crim"])
Y = boston["crim"]
print(X)
print(Y)

       zn  indus  chas    nox     rm   age     dis  rad  tax  ptratio  lstat  \
0    18.0   2.31     0  0.538  6.575  65.2  4.0900    1  296     15.3   4.98   
1     0.0   7.07     0  0.469  6.421  78.9  4.9671    2  242     17.8   9.14   
2     0.0   7.07     0  0.469  7.185  61.1  4.9671    2  242     17.8   4.03   
3     0.0   2.18     0  0.458  6.998  45.8  6.0622    3  222     18.7   2.94   
4     0.0   2.18     0  0.458  7.147  54.2  6.0622    3  222     18.7   5.33   
..    ...    ...   ...    ...    ...   ...     ...  ...  ...      ...    ...   
501   0.0  11.93     0  0.573  6.593  69.1  2.4786    1  273     21.0   9.67   
502   0.0  11.93     0  0.573  6.120  76.7  2.2875    1  273     21.0   9.08   
503   0.0  11.93     0  0.573  6.976  91.0  2.1675    1  273     21.0   5.64   
504   0.0  11.93     0  0.573  6.794  89.3  2.3889    1  273     21.0   6.48   
505   0.0  11.93     0  0.573  6.030  80.8  2.5050    1  273     21.0   7.88   

     medv  
0    24.0  
1    21.6  
2  

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=0)

# (a) 

In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Ridge

In [6]:
import numpy as np
import sklearn.linear_model as skl
from sklearn.model_selection import GridSearchCV
alpha = np.logspace(-4, 4, 100)
ridge = skl.Ridge()
grid_cv = GridSearchCV(ridge, param_grid={'alpha':alpha}, cv=5)
grid_cv.fit(X_train_s, Y_train)
grid_cv.best_estimator_

Ridge(alpha=np.float64(37.649358067924716))

In [7]:
grid_cv.best_params_

{'alpha': np.float64(37.649358067924716)}

In [8]:
from sklearn import metrics
Y_ridge_pred = grid_cv.predict(X_test_s)
mse_ridge = metrics.mean_squared_error(Y_test, Y_ridge_pred)
mse_ridge

52.27346426043761

## Lasso

In [9]:
lasso = skl.Lasso()
lasso_grid_cv = GridSearchCV(lasso, param_grid={'alpha':alpha}, cv=5)
lasso_grid_cv.fit(X_train_s, Y_train)
print(lasso_grid_cv.best_params_)

{'alpha': np.float64(0.7564633275546291)}


In [10]:
Y_lasso_pred = lasso_grid_cv.predict(X_test_s)
mse_lasso = metrics.mean_squared_error(Y_test, Y_lasso_pred)
mse_lasso

54.671874666030014

## PCR

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
pcr_pipeline = Pipeline(
    steps=[
        ('pca', PCA()),
        ('regression', skl.LinearRegression())
    ]
)
n_components = range(1, 13)
pcr_grid_cv = GridSearchCV(pcr_pipeline, param_grid={'pca__n_components':n_components}, cv=5)
pcr_grid_cv.fit(X_train_s, Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('pca', PCA()),
                                       ('regression', LinearRegression())]),
             param_grid={'pca__n_components': range(1, 13)})

In [13]:
Y_pcr_pred = pcr_grid_cv.predict(X_test_s)
mse_pcr = metrics.mean_squared_error(Y_test, Y_pcr_pred)
mse_pcr

51.36409047543869

# Linear Regression

In [15]:
linear_model = skl.LinearRegression()
linear_model.fit(X_train_s, Y_train)
Y_linear_model_pred = linear_model.predict(X_test_s)
mse_linear_model = metrics.mean_squared_error(Y_test, Y_linear_model_pred)
mse_linear_model

51.36409047543869

# Forward Selection

In [20]:
from sklearn.feature_selection import SequentialFeatureSelector
sfs_pipeline = Pipeline([
    ('selector', SequentialFeatureSelector(skl.LinearRegression(), direction='forward', cv = 5)),
    ('regression', skl.LinearRegression())
])

sfs_grid_cv = GridSearchCV(sfs_pipeline, param_grid={'selector__n_features_to_select':range(1, 13)})
sfs_grid_cv.fit(X_train_s, Y_train)
Y_sfs_pred = sfs_grid_cv.predict(X_test_s)
mse_sfs = metrics.mean_squared_error(Y_test, Y_sfs_pred)

/home/sonu/miniconda3/envs/islp/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
5 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/sonu/miniconda3/envs/islp/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sonu/miniconda3/envs/islp/lib/python3.13/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/sonu/miniconda3/envs/islp/lib/python3.13/site-package

In [21]:
mse_sfs

52.357000923533654

# (b)

# (c)